In [ ]:
import numpy as np
import random

GRID_SIZE = 5
GAMMA = 0.65
ALPHA = 0.1
EPISODES = 2000
EPSILON = 0.1

START_STATE = (0, 0)
GOAL_STATE = (4, 4)
OBSTACLES = [(1, 1), (2, 2), (3, 1)]

ACTIONS = [0, 1, 2, 3]

In [ ]:
def get_state_idx(state):
    return state[0] * GRID_SIZE + state[1]

def step(state, action):
    r, c = state

    if action == 0 and r > 0: r -= 1    # Up
    elif action == 1 and r < 4: r += 1  # Down
    elif action == 2 and c > 0: c -= 1  # Left
    elif action == 3 and c < 4: c += 1  # Right

    next_s = (r, c)

    if next_s == GOAL_STATE:
        return next_s, 10, True
    elif next_s in OBSTACLES:
        return next_s, -10, True
    else:
        return next_s, -1, False

In [ ]:
q_table = np.zeros((GRID_SIZE * GRID_SIZE, len(ACTIONS)))

print("Training started...")

for ep in range(EPISODES):
    current_state = START_STATE
    done = False

    while not done:
        s_idx = get_state_idx(current_state)

        if random.uniform(0, 1) < EPSILON:
            action = random.choice(ACTIONS)
        else:
            action = np.argmax(q_table[s_idx])

        next_state, reward, done = step(current_state, action)
        ns_idx = get_state_idx(next_state)

        best_next_q = np.max(q_table[ns_idx])
        q_table[s_idx, action] += ALPHA * (reward + GAMMA * best_next_q - q_table[s_idx, action])

        current_state = next_state

print("Training finished.")

def get_best_path():
    path = []
    curr = START_STATE
    path.append(curr)
    max_steps = 20

    while curr != GOAL_STATE and max_steps > 0:
        s_idx = get_state_idx(curr)
        best_action = np.argmax(q_table[s_idx])
        curr, _, _ = step(curr, best_action)
        path.append(curr)
        max_steps -= 1
    return path

optimal_path = get_best_path()
print(f"The optimal path found by the agent: {optimal_path}")

Training started...
Training finished.
The optimal path found by the agent: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (1, 4), (2, 4), (3, 4), (4, 4)]


In [ ]:
grid_view = np.zeros((5, 5), dtype=str)
grid_view[:] = '.'
for obs in OBSTACLES: grid_view[obs] = 'X'
grid_view[GOAL_STATE] = 'G'

for p in optimal_path:
    if p != GOAL_STATE and p not in OBSTACLES:
        grid_view[p] = '*'

print("Grid Map (* is Path, X is Obstacle, G is Goal):")
print(grid_view)

Grid Map (* is Path, X is Obstacle, G is Goal):
[['*' '*' '*' '*' '.']
 ['.' 'X' '.' '*' '*']
 ['.' '.' 'X' '.' '*']
 ['.' 'X' '.' '.' '*']
 ['.' '.' '.' '.' 'G']]
